Imports


In [70]:
import pandas as pd
import numpy as np
import pickle, json

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut

import warnings
warnings.filterwarnings("ignore")

# Paths

In [71]:
BASE_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/data/raw/"

MODEL_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/src/interactive_interface/model.pkl"
LOOKUP_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/src/interactive_interface/opponent_lookup.json"

RESULTS_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/data/raw/model_results_combined.csv"


# Settings

In [72]:
STADIUM_CAPACITY = 10_000
USE_SIGMOID_TARGET = False
USE_LOG_TARGET = True
LOG_CLIP_MIN, LOG_CLIP_MAX = 6.0, 10.5


# Final 21 features 

In [73]:
FEATURES_21 = [
    "attendance_lag_1",
    "attendance_roll_3",
    "attendance_roll_5",
    "tickets_sold_b2c",
    "tickets_sold_b2b",
    "opponent_avg_attendance_raw",
    "is_top_opponent",
    "points_last_5",
    "goal_diff_last_5",
    "form_x_opponent",
    "match_attractiveness",
    "season_progress",
    "season_enc",
    "is_playoff",
    "has_promotion",
    "pct_free_tickets",
    "kickoff_hour",
    "weather_rain_mm",
    "ohl_interest",
    "article_count_7d",
    "lag_x_form",
]

TARGET = "tickets_scanned"

# Helper functions for targets


In [74]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def normalize_attendance(y, capacity=STADIUM_CAPACITY):
    rate = y / capacity
    centered = rate - 0.5
    return sigmoid(centered * 6)


def inverse_sigmoid_attendance(y_sig, capacity=STADIUM_CAPACITY):
    clipped = np.clip(y_sig, 1e-6, 1 - 1e-6)
    centered = np.log(clipped / (1 - clipped)) / 6
    return (centered + 0.5) * capacity

# Load already prepared data

In [75]:
df_match = pd.read_csv(BASE_PATH + "gold_match.csv")
df_trends = pd.read_csv(BASE_PATH + "gold_google_trends_daily.csv")
df_tickets = pd.read_csv(BASE_PATH + "gold_match_tickets.csv")
df_context = pd.read_csv(BASE_PATH + "gold_match_context.csv")
df_goals = pd.read_csv(BASE_PATH + "gold_match_goals.csv")
df_articles = pd.read_csv(
    BASE_PATH + "gold_belga_press_articles.csv",
    escapechar="\\",
    on_bad_lines="skip",
)


#  REBUILD df

In [76]:
df = df_match[df_match["is_home_match"] == True].copy()
df = df.merge(df_tickets, on="match_id", how="left")
df = df.merge(df_context, on="match_id", how="left")
df["match_date"] = df["match_date_x"]
df = df.drop(columns=["match_date_x", "match_date_y"], errors="ignore")

df_trends_agg = df_trends.groupby("match_id")["ohl_interest"].mean().reset_index()
df = df.merge(df_trends_agg, on="match_id", how="left")

df_articles_agg = df_articles.groupby("match_id").size().reset_index(name="article_count")
df = df.merge(df_articles_agg, on="match_id", how="left")
df["article_count"] = df["article_count"].fillna(0)

df_articles["date"] = pd.to_datetime(df_articles["date"])
art_pre = []
for _, row in df[["match_id", "match_date"]].iterrows():
    window = df_articles[
        (df_articles["match_id"] == row["match_id"])
        & (df_articles["days_to_match"].between(-7, -1))
    ]
    art_pre.append({"match_id": row["match_id"], "article_count_7d": len(window)})

df_art_pre = pd.DataFrame(art_pre)
df = df.merge(df_art_pre, on="match_id", how="left")
df["article_count_7d"] = df["article_count_7d"].fillna(0)

df = df.sort_values("match_date").reset_index(drop=True)

# Feature engineering

In [77]:
df["kickoff_hour"] = pd.to_datetime(df["kickoff_time_local"], format="%H:%M:%S").dt.hour

points_map = {"W": 3, "D": 1, "L": 0}
df["points"] = df["result_home"].map(points_map)
df["points_last_5"] = (
    df["points"].rolling(5).sum().shift(1).fillna(df["points"].mean() * 5)
)
df["goal_diff"] = df["goals_home_ft"] - df["goals_away_ft"]
df["goal_diff_last_5"] = df["goal_diff"].rolling(5).sum().shift(1).fillna(0)

df["matchday"] = pd.to_numeric(df["matchday"], errors="coerce")
df["season_progress"] = df["matchday"] / df["matchday"].max()

df["form_strength_norm"] = (
    (df["points_last_5"] - df["points_last_5"].min())
    / (df["points_last_5"].max() - df["points_last_5"].min())
)
df["match_importance"] = 0.5 * df["form_strength_norm"] + 0.5 * df["season_progress"]

df["opponent"] = df["away_team"]
df["opponent_freq"] = df["opponent"].map(df["opponent"].value_counts())

top_teams = ["Club Brugge", "Anderlecht", "STVV", "KV Mechelen", "Westerlo"]
df["is_top_opponent"] = df["away_team"].isin(top_teams).astype(int)

df["opponent_strength_norm"] = (
    (df["opponent_freq"] - df["opponent_freq"].min())
    / (df["opponent_freq"].max() - df["opponent_freq"].min())
)

df["match_attractiveness"] = (
    0.4 * df["match_importance"]
    + 0.4 * df["opponent_strength_norm"]
    + 0.2 * df["is_top_opponent"]
)
df["form_x_opponent"] = df["points_last_5"] * df["is_top_opponent"]

df["opponent_avg_attendance_raw"] = df.groupby("away_team")["tickets_sold_total"].transform(
    "mean"
)

df["attendance_lag_1"] = df["tickets_scanned"].shift(1).fillna(df["tickets_scanned"].mean())
df["has_promotion"] = df["has_promotion"].astype(int)

df["is_school_holiday_flanders"] = df["is_school_holiday_flanders"].astype(int)
df["is_public_holiday"] = df["is_public_holiday"].astype(int)
df["is_midweek"] = df["is_midweek"].astype(int)

df["is_playoff"] = df["stage"].str.contains("Playoff|Play-off", case=False, na=False).astype(
    int
)

result_map = {"W": 1, "D": 0, "L": -1}
df["last_result_vs_opponent_enc"] = df["last_result_vs_opponent"].map(result_map).fillna(0)

df["ohl_interest"] = df["ohl_interest"].fillna(df["ohl_interest"].median())
df["weather_score"] = df["weather_score"].fillna(df["weather_score"].median())
df["weather_rain_mm"] = df["weather_rain_mm"].fillna(df["weather_rain_mm"].median())

df["promo_x_top_opponent"] = df["has_promotion"] * df["is_top_opponent"]
df["lag_x_form"] = df["attendance_lag_1"] * df["points_last_5"]
df["rain_x_weekend"] = df["weather_rain_mm"] * df["is_weekend"]
df["playoff_x_top"] = df["is_playoff"] * df["is_top_opponent"]

df["attendance_roll_3"] = (
    df["tickets_scanned"].rolling(3).mean().shift(1).fillna(df["tickets_scanned"].mean())
)
df["attendance_roll_5"] = (
    df["tickets_scanned"].rolling(5).mean().shift(1).fillna(df["tickets_scanned"].mean())
)

df["season_enc"] = pd.factorize(df["season"])[0]


# Build modeling dataset with FINAL 21 features

In [78]:

features_combined = [f for f in FEATURES_21 if f in df.columns]

df_model = df[features_combined + [TARGET, "away_team"]].dropna()
X = df_model[features_combined]
y = df_model[TARGET]

if USE_LOG_TARGET:
    y_fit = np.log(y)
elif USE_SIGMOID_TARGET:
    y_fit = normalize_attendance(y)
else:
    y_fit = y

print(f"Dataset: {df_model.shape[0]} rows, {len(features_combined)} features")
print("Features:", features_combined)

Dataset: 71 rows, 21 features
Features: ['attendance_lag_1', 'attendance_roll_3', 'attendance_roll_5', 'tickets_sold_b2c', 'tickets_sold_b2b', 'opponent_avg_attendance_raw', 'is_top_opponent', 'points_last_5', 'goal_diff_last_5', 'form_x_opponent', 'match_attractiveness', 'season_progress', 'season_enc', 'is_playoff', 'has_promotion', 'pct_free_tickets', 'kickoff_hour', 'weather_rain_mm', 'ohl_interest', 'article_count_7d', 'lag_x_form']


# Train Gradient Boosting with LOOCV

In [79]:
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.7,
    min_samples_leaf=8,
    random_state=42,
)

loo = LeaveOneOut()
y_true_list, y_pred_list = [], []

scaler = StandardScaler()

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train_fold = y_fit.iloc[train_idx]

    # opponent_avg_attendance_raw recalculated in each fold
    if "opponent_avg_attendance_raw" in features_combined:
        opp_avg = df_model.iloc[train_idx].groupby("away_team")[TARGET].mean()
        global_avg_fold = y.iloc[train_idx].mean()
        opp_test = df_model.iloc[test_idx]["away_team"].values[0]
        X_train = X_train.copy()
        X_test = X_test.copy()
        X_train["opponent_avg_attendance_raw"] = df_model.iloc[train_idx]["away_team"].map(
            opp_avg
        )
        X_test["opponent_avg_attendance_raw"] = opp_avg.get(opp_test, global_avg_fold)

    X_tr_s = scaler.fit_transform(X_train)
    X_te_s = scaler.transform(X_test)

    model.fit(X_tr_s, y_train_fold)
    pred = model.predict(X_te_s)[0]

    if USE_LOG_TARGET:
        pred = np.exp(np.clip(pred, LOG_CLIP_MIN, LOG_CLIP_MAX))
    elif USE_SIGMOID_TARGET:
        pred = inverse_sigmoid_attendance(pred)

    y_true_list.append(y.iloc[test_idx].values[0])
    y_pred_list.append(pred)

# Metrics
mae = mean_absolute_error(y_true_list, y_pred_list)
rmse = np.sqrt(mean_squared_error(y_true_list, y_pred_list))
r2 = r2_score(y_true_list, y_pred_list)
mape = np.mean(
    np.abs((np.array(y_true_list) - np.array(y_pred_list)) / np.array(y_true_list))
) * 100

results_df = pd.DataFrame(
    {
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2],
        "MAPE": [mape],
    },
    index=["GradientBoosting"],
)

print(results_df.round(2))


                     MAE     RMSE    R2   MAPE
GradientBoosting  875.49  1144.74  0.67  13.89


# Refit on full data and save artifacts

In [81]:
scaler_full = StandardScaler()
X_full_s = scaler_full.fit_transform(X)
model.fit(X_full_s, y_fit)

with open(MODEL_PATH, "wb") as f:
    pickle.dump(
        {
            "model": model,
            "scaler": scaler_full,
            "features": features_combined,
            "log_target": USE_LOG_TARGET,
            "sigmoid": USE_SIGMOID_TARGET,
            "capacity": STADIUM_CAPACITY,
            "log_clip": (LOG_CLIP_MIN, LOG_CLIP_MAX),
        },
        f,
    )

opp_lookup = df_model.groupby("away_team")[TARGET].mean().round(0).to_dict()
opp_lookup["__global_avg__"] = float(y.mean())
with open(LOOKUP_PATH, "w") as f:
    json.dump(opp_lookup, f, indent=2)

results_df.to_csv(RESULTS_PATH)
print("GradientBoosting")
print("model.pkl, opponent_lookup.json, model_results_combined.csv saved.")

GradientBoosting
model.pkl, opponent_lookup.json, model_results_combined.csv saved.
